# Store Sales - Time Series Forecasting

In [ ]:
import pandas as pd
import numpy as np

import re
import sys

import matplotlib.pyplot as plt
import seaborn as sns

import missingno as msno
import scipy.stats as stats
from patsy import dmatrices
import statsmodels.api as sm 

from sklearn.model_selection import train_test_split
from sklearn.model_selection import ParameterGrid

from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Setting up the environment
# plt.style.use('seaborn-v0_8-darkgrid')
sns.set_style("whitegrid")

if sys.platform == 'win32':
    print('Win')
    plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei']
elif sys.platform == 'darwin':
    print('Mac')
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']

plt.rcParams['axes.unicode_minus']=False
# setting pandas display options
pd.set_option('display.max_columns', None)  # or 1000
pd.set_option('display.max_rows', None)  # or 1000
pd.set_option('display.max_colwidth', None)  # or 199

# pd.set_option('display.max_columns', 20)  # or 1000
# pd.set_option('display.max_rows', 80)  # or 1000
# pd.set_option('display.max_colwidth', 20)  # or 199

In [ ]:
# path = './'
path = '/kaggle/input/store-sales-time-series-forecasting'

## Data Pipeline


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
import pandas as pd

# Load Data
class LoadData(BaseEstimator, TransformerMixin):
    def __init__(self, train_path=path+'/train.csv', store_path=path+'/stores.csv'):
        self.train_path = train_path
        self.store_path = store_path

    def fit(self, X=None, y=None):
        return self

    def transform(self, X=None):
        train_df = pd.read_csv(self.train_path)
        stores_df = pd.read_csv(self.store_path)
        train_df['date'] = pd.to_datetime(train_df['date'])
        return {'train_df': train_df, 'stores_df': stores_df}


# Generate Sales Pivot Table
class PivotSales(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        grouped = X['train_df'].groupby(['date', 'store_nbr', 'family'])['sales'].sum().reset_index()
        pivot_sales = grouped.pivot(index=['date', 'store_nbr'], columns='family', values='sales').reset_index()
        pivot_sales = pivot_sales.fillna(0)
        X['pivot_sales'] = pivot_sales
        return X


# Generate Promotions Pivot Table
class PivotPromotions(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        grouped_promo = X['train_df'].groupby(['date', 'store_nbr', 'family'])['onpromotion'].sum().reset_index()
        pivot_promo = grouped_promo.pivot(index=['date', 'store_nbr'], columns='family', values='onpromotion').reset_index()
        pivot_promo = pivot_promo.fillna(0)

        rename_dict = {
            col: f'onpromotion_{col}' for col in pivot_promo.columns if col not in ['date', 'store_nbr']
        }
        pivot_promo = pivot_promo.rename(columns=rename_dict)
        X['pivot_promo'] = pivot_promo
        return X


# Generate Sales and Promotions Merge
class MergeSalesPromo(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        merged = X['pivot_sales'].merge(X['pivot_promo'], on=['date', 'store_nbr'], how='inner')
        X['family_sales_df'] = merged
        return X


# Merge Store Information
class MergeStoreInfo(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        merged = X['family_sales_df'].merge(X['stores_df'], on='store_nbr', how='left')
        return merged

class AddHolidayFeatures(BaseEstimator, TransformerMixin):
    def __init__(self, holiday_path=path+'/holidays_events.csv'):
        self.holiday_path = holiday_path

    def fit(self, X, y=None):
        return self

    def transform(self, X: pd.DataFrame):
        holidays = pd.read_csv(self.holiday_path)
        holidays = holidays[holidays['transferred'] == False].copy()
        holidays['date'] = pd.to_datetime(holidays['date'])

        
        unique_keys = holidays[['type', 'locale', 'locale_name']].drop_duplicates()

        
        for _, row in unique_keys.iterrows():
            dummy_col = f"holiday_{row['locale']}_{row['type']}".replace(" ", "_")
            X[dummy_col] = 0

        
        for _, row in holidays.iterrows():
            mask = X['date'] == row['date']

            if row['locale'] == 'National':
                X.loc[mask, f"holiday_National_{row['type']}".replace(" ", "_")] = 1

            elif row['locale'] == 'Regional':
                mask &= X['state'] == row['locale_name']
                X.loc[mask, f"holiday_Regional_{row['type']}".replace(" ", "_")] = 1

            elif row['locale'] == 'Local':
                mask &= X['city'] == row['locale_name']
                X.loc[mask, f"holiday_Local_{row['type']}".replace(" ", "_")] = 1

        return X

class AddOilFeature(BaseEstimator, TransformerMixin):
    def __init__(self, oil_path=path+'/oil.csv'):
        self.oil_path = oil_path

    def fit(self, X, y=None):
        return self

    def transform(self, X: pd.DataFrame):
        oil_df = pd.read_csv(self.oil_path)
        oil_df['date'] = pd.to_datetime(oil_df['date'])

        oil_df['dcoilwtico_ma7'] = oil_df['dcoilwtico'].rolling(window=7, center=True, min_periods=1).mean()


        oil_df['dcoilwtico'] = oil_df['dcoilwtico'].fillna(oil_df['dcoilwtico_ma7'])

        oil_df['dcoilwtico'] = oil_df['dcoilwtico'].fillna(method='bfill').fillna(method='ffill')

        oil_df = oil_df.drop(columns=['dcoilwtico_ma7'])


        X = X.merge(oil_df, on='date', how='left')
        return X



class FeatureEngineering(BaseEstimator, TransformerMixin):
    def __init__(self, base_target='GROCERY I'):
        self.base_target = base_target

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        df = df.sort_values(['store_nbr', 'date'])

        # Time Features
        df['year'] = df['date'].dt.year
        df['month'] = df['date'].dt.month
        df['day'] = df['date'].dt.day

        df['year_month'] = df['date'].dt.to_period('M')
        
        df['dayofweek'] = df['date'].dt.dayofweek
        df['weekofyear'] = df['date'].dt.isocalendar().week.astype(int)
        df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)

        # Lag Features
        for lag in [1, 7, 14]:
            df[f'{self.base_target}_lag_{lag}'] = df.groupby('store_nbr')[self.base_target].shift(lag)

        # Moving Average Features
        for window in [7, 14]:
            df[f'{self.base_target}_ma_{window}'] = df.groupby('store_nbr')[self.base_target].transform(lambda x: x.shift(1).rolling(window).mean())

        return df


# === Create Pipeline ===
data_pipeline = Pipeline(steps=[
    ('load_data', LoadData(path+'/train.csv', path+'/stores.csv')),
    ('pivot_sales', PivotSales()),
    ('pivot_promotions', PivotPromotions()),
    ('merge_sales_promo', MergeSalesPromo()),
    ('merge_store_info', MergeStoreInfo()),
    ('add_holidays', AddHolidayFeatures(path+'/holidays_events.csv')),
    ('add_oil', AddOilFeature(path+'/oil.csv'))
])

family_sales_df = data_pipeline.fit_transform(None)


display("Family Sales DataFrame")
display(family_sales_df.head())
display(family_sales_df.shape)


# EDA & Visualization

### Tread by Store & Loaction

In [ ]:
# store_nbr 
# Get family sales columns

family_cols = [col for col in family_sales_df.columns if col not in ['date', 'store_nbr', 'city', 'state', 'cluster', 'type', 'dcoilwtico'] and not col.startswith('holiday') and not col.startswith('onpromotion')]

store_family_sales = family_sales_df.groupby('store_nbr')[family_cols].mean()

plt.figure(figsize=(18, 8))
sns.heatmap(store_family_sales, cmap='YlGnBu')
plt.title('Average Sales per Family by Store')
plt.xlabel('Family')
plt.ylabel('Store Number')
plt.xticks(rotation=-45, ha='left')
plt.yticks(rotation=0)
plt.show()


In [ ]:


df_plot = family_sales_df.copy()
df_plot['year_week'] = df_plot['date'].dt.to_period('W')
df_plot['year_week'] = df_plot['year_week'].dt.to_timestamp()


sns.set_style("whitegrid")
fig, axes = plt.subplots(3, 1, figsize=(16, 18), sharex=True)

# difference City
city_weekly = (
    df_plot.groupby(['city','year_week'])[family_cols]
    .sum()
    .sum(axis=1)
    .reset_index(name="sales")
)

sns.lineplot(
    data=city_weekly,
    x="year_week", y="sales", hue="city",
    palette="tab10", ax=axes[0]
)
axes[0].set_title("Weekly Sales Trend by City", fontsize=14)
axes[0].set_ylabel("Total Sales")
axes[0].legend(bbox_to_anchor=(1.02, 1), loc='upper left')

# diff Type
type_weekly = (
    df_plot.groupby(['type','year_week'])[family_cols]
    .sum()
    .sum(axis=1)
    .reset_index(name="sales")
)

sns.lineplot(
    data=type_weekly,
    x="year_week", y="sales", hue="type",
    palette="tab10", ax=axes[1]
)
axes[1].set_title("Weekly Sales Trend by Store Type", fontsize=14)
axes[1].set_ylabel("Total Sales")
axes[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left')

# difference Cluster
cluster_weekly = (
    df_plot.groupby(['cluster','year_week'])[family_cols]
    .sum()
    .sum(axis=1)
    .reset_index(name="sales")
)

sns.lineplot(
    data=cluster_weekly,
    x="year_week", y="sales", hue="cluster",
    palette="tab10", ax=axes[2]
)
axes[2].set_title("Weekly Sales Trend by Cluster", fontsize=14)
axes[2].set_ylabel("Total Sales")
axes[2].legend(bbox_to_anchor=(1.02, 1), loc='upper left')


plt.xlabel("Year-Week")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:


df_plot = family_sales_df.copy()
df_plot['year_month'] = df_plot['date'].dt.to_period('M')


city_cluster_monthly = (
    df_plot.groupby(['city','cluster','year_month'])[family_cols]
    .sum()
    .sum(axis=1)
    .reset_index(name="sales")
)
city_cluster_monthly['year_month'] = city_cluster_monthly['year_month'].dt.to_timestamp()


g = sns.FacetGrid(
    city_cluster_monthly,
    col="city",
    hue="cluster",          
    col_wrap=3,
    height=3.5,
    sharey=False
)

g.map_dataframe(
    sns.lineplot,
    x="year_month",
    y="sales"
)

g.add_legend(title="Cluster")
g.set_titles(col_template="{col_name}")
for ax in g.axes.flatten():
    ax.tick_params(axis='x', rotation=45)
    ax.set_ylabel("Total Sales")
    ax.set_xlabel("Year-Month")

plt.subplots_adjust(top=0.9)
g.fig.suptitle('Monthly Sales Trend by City × Cluster', fontsize=16)
plt.show()


### Trend by Family Goods

In [ ]:
plt.figure(figsize=(18, 8))
family_sales_melted_df = family_sales_df.melt(id_vars=['store_nbr', 'date'], value_vars=family_cols, var_name='Family', value_name='Sales')

sns.boxplot(data=family_sales_melted_df, x='Family', y='Sales')
plt.xticks(rotation=-45, ha='left')
plt.title('Distribution of Sales by Family (All Stores)')
plt.show()

df_plot = family_sales_df.copy()
df_plot['year_month'] = df_plot['date'].dt.to_period('M')
monthly_sales_family = df_plot.groupby('year_month')[family_cols].sum()

# Corr  Matrix
corr_matrix = monthly_sales_family.corr()

plt.figure(figsize=(14,10))

sns.heatmap(corr_matrix, cmap="coolwarm", annot=False)
plt.xticks(rotation=-45, ha='left')
plt.title("Correlation of Sales Between Families")
plt.show()


In [ ]:

family_list = [col for col in family_sales_df.columns if col not in ['date','year_month','store_nbr','city','state','cluster','type','dcoilwtico'] 
                and not col.startswith('onpromotion') 
                and not col.startswith('holiday')]

df_plot = family_sales_df.copy()
df_plot['year_month'] = df_plot['date'].dt.to_period('M')

# Relation between Promotion and Sales
corr_results = []
for fam in family_list:
    promo_col = f'onpromotion_{fam}'
    if promo_col in df_plot.columns:
        monthly_data = df_plot.groupby('year_month').agg({
            fam: 'sum',
            promo_col: 'sum'
        }).reset_index()
        
        corr_value = monthly_data[fam].corr(monthly_data[promo_col])
        corr_results.append((fam, corr_value))

# 轉成 DataFrame
corr_df = pd.DataFrame(corr_results, columns=['Family', 'Correlation']).sort_values('Correlation', ascending=False)

# 顯示結果
plt.figure(figsize=(10,6))
sns.barplot(data=corr_df, x='Correlation', y='Family')
plt.title('Promotion-Sales Correlation by Family')
plt.xlabel('Correlation Coefficient')
plt.ylabel('Family')
plt.show()




In [ ]:
# Monthly Sales Trend by Family

monthly_sales_family = df_plot.groupby('year_month')[family_cols].sum()

plt.figure(figsize=(14,8))
for col in monthly_sales_family.columns:
    plt.plot(monthly_sales_family.index.astype(str), monthly_sales_family[col], label=col)

plt.xticks(rotation=45)
plt.title('Monthly Sales Trend by Family')
plt.xlabel('Year-Month')
plt.ylabel('Total Sales')
plt.legend(loc='upper left', bbox_to_anchor=(1,1))
plt.grid(True)
plt.show()


In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram

plt.figure(figsize=(12, 6))
Z = linkage(corr_matrix, method='ward')
dendrogram(Z, labels=corr_matrix.columns, leaf_rotation=-45, leaf_font_size=10)
plt.xticks(rotation=-45, ha='left')
plt.title("Hierarchical Clustering of Families Based on Sales")
plt.show()

### Trend About Time Series

In [ ]:

df_plot = family_sales_df.copy()
df_plot['year_week'] = df_plot['date'].dt.to_period('W')

weekly_sales_total = df_plot.groupby('year_week')[family_cols].sum().sum(axis=1)
weekly_sales_total.index = weekly_sales_total.index.to_timestamp()


window_size = 12
rolling_mean = weekly_sales_total.rolling(window=window_size, center=True).mean()
rolling_std = weekly_sales_total.rolling(window=window_size, center=True).std()

upper_bound = rolling_mean + rolling_std
lower_bound = rolling_mean - rolling_std


mean_sales = weekly_sales_total.mean()


holidays = pd.read_csv(path+'/holidays_events.csv')
holidays = holidays[
    (holidays['transferred'] == False) &
    (holidays['locale'].isin(['National', 'Regional'])) &
    (pd.to_datetime(holidays['date']).dt.year >= 2013)
].copy()
holidays['date'] = pd.to_datetime(holidays['date'])
holidays['year_week'] = holidays['date'].dt.to_period('W')

holidays_week = holidays.drop_duplicates(['year_week', 'locale', 'locale_name'])


fig, ax = plt.subplots(figsize=(18,10))
ax.plot(weekly_sales_total.index, weekly_sales_total.values, marker='o', label='Total Sales')


ax.fill_between(
    weekly_sales_total.index,
    lower_bound,
    upper_bound,
    color='green',
    alpha=0.1,
    label='±1 Std Dev'
)

label_y_offset = -0.08  # base y-offset for holiday labels

for _, row in holidays_week.iterrows():
    week_date = row['year_week'].to_timestamp()
    
    if row['locale'] == 'National':
        color = 'red'
        label_text = f"{row['description']} (Nat)"
    else:
        color = 'blue'
        label_text = f"{row['description']} ({row['locale_name']})"
    
    # 
    ax.axvline(x=week_date, color=color, linestyle='--', alpha=0.3)
    
    ax.text(
        week_date,
        label_y_offset - 0.05,
        label_text,
        rotation=-55,
        fontsize=8,
        fontweight='bold',
        color=color,
        ha='left', va='top',
        transform=ax.get_xaxis_transform(),  
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.6, boxstyle='round,pad=0.1')
    )


ax.set_xticklabels(ax.get_xticklabels(), rotation=-45, ha='left')
ax.set_title('Weekly Sales Trend (2013+) with Holidays Labels Below X-axis')
ax.set_xlabel('Year-Week')
ax.set_ylabel('Total Sales')
ax.grid(True)
ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:

family_sales_df['dayofweek'] = family_sales_df['date'].dt.dayofweek  # 0=Monday

weekly_sales = family_sales_df.groupby('dayofweek')[family_cols].sum().sum(axis=1)

plt.figure(figsize=(8,4))
sns.barplot(x=weekly_sales.index, y=weekly_sales.values)
plt.title('Average Sales by Day of Week (All Families)')
plt.xlabel('Day of Week')
plt.ylabel('Total Sales')
plt.show()

family_sales_df['month'] = family_sales_df['date'].dt.month

monthly_sales = family_sales_df.groupby('month')[family_cols].sum().sum(axis=1)

plt.figure(figsize=(8,4))
sns.barplot(x=monthly_sales.index, y=monthly_sales.values)
plt.title('Average Sales by Month (All Families)')
plt.xlabel('Month')
plt.ylabel('Total Sales')
plt.show()


from statsmodels.tsa.seasonal import seasonal_decompose


df_plot = family_sales_df.copy()
df_plot['year_month'] = df_plot['date'].dt.to_period('M')
monthly_sales_series = df_plot.groupby('year_month')[family_cols].sum().sum(axis=1)


monthly_sales_series.index = monthly_sales_series.index.to_timestamp()


result = seasonal_decompose(monthly_sales_series, model='additive', period=12)


fig = result.plot()
fig.set_size_inches(12, 9)   


fig.suptitle("Seasonal Decomposition of Total Monthly Sales", fontsize=16, fontweight="bold", y=0.95)
fig.axes[0].set_ylabel("Observed", fontsize=12)
fig.axes[1].set_ylabel("Trend", fontsize=12)
fig.axes[2].set_ylabel("Seasonal", fontsize=12)
fig.axes[3].set_ylabel("Residual", fontsize=12)


for ax in fig.axes:
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()



In [ ]:
from statsmodels.tsa.stattools import acf, pacf
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd


df_plot = family_sales_df.copy()
df_plot['year_week'] = df_plot['date'].dt.to_period('W')
df_plot['year_week'] = df_plot['year_week'].dt.to_timestamp()

lags = 12
acf_matrix = pd.DataFrame(index=family_cols, columns=[f"Lag_{i}" for i in range(1, lags+1)])
pacf_matrix = pd.DataFrame(index=family_cols, columns=[f"Lag_{i}" for i in range(1, lags+1)])

# Calculate ACF and PACF
for fam in family_cols:
    weekly_sales = df_plot.groupby('year_week')[fam].sum()
    acf_values = acf(weekly_sales, nlags=lags, fft=False)[1:]   # miss Lag=0
    pacf_values = pacf(weekly_sales, nlags=lags, method='ywm')[1:] # miss Lag=0
    
    acf_matrix.loc[fam] = acf_values
    pacf_matrix.loc[fam] = pacf_values

acf_matrix = acf_matrix.astype(float)
pacf_matrix = pacf_matrix.astype(float)


fig, axes = plt.subplots(2, 1, figsize=(14, 18))

# ACF Hot map
sns.heatmap(acf_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[0])
axes[0].set_title("ACF (Lag=1~12) for All Families (Weekly Sales)", fontsize=16)
axes[0].set_xlabel("Lag")
axes[0].set_ylabel("Family")

# PACF Hot map
sns.heatmap(pacf_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1])
axes[1].set_title("PACF (Lag=1~12) for All Families (Weekly Sales)", fontsize=16)
axes[1].set_xlabel("Lag")
axes[1].set_ylabel("Family")

plt.tight_layout()
plt.show()


# Model

###  ARIMA 

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller, acf, pacf
from tqdm import tqdm


train_df = pd.read_csv(path+'/train.csv')
test_df = pd.read_csv(path+'/test.csv')

train_df['date'] = pd.to_datetime(train_df['date'])
test_df['date'] = pd.to_datetime(test_df['date'])

# Decide d
def get_diff_order(series, alpha=0.05):
    d = 0
    while True:
        p_value = adfuller(series.dropna())[1]
        if p_value > alpha:
            d += 1
            series = series.diff().dropna()
        else:
            break
    return min(d, 2)  


# Decide p, q
def get_pq_orders(series, nlags=12):
    # PACF to choose  p
    pacf_vals = pacf(series.dropna(), nlags=nlags, method='ywm')
    p_candidates = np.where(np.abs(pacf_vals) > 1.96/np.sqrt(len(series)))[0]
    p = int(p_candidates[1]) if len(p_candidates) > 1 else 1

    # ACF to choose  q
    acf_vals = acf(series.dropna(), nlags=nlags)
    q_candidates = np.where(np.abs(acf_vals) > 1.96/np.sqrt(len(series)))[0]
    q = int(q_candidates[1]) if len(q_candidates) > 1 else 1

    return min(p, 3), min(q, 3)  # 

# Save predictions
pred_results = []

store_list = train_df['store_nbr'].unique()
family_list = train_df['family'].unique()

for store in tqdm(store_list, desc="Stores"):
    for fam in family_list:

        sub_train = train_df[(train_df['store_nbr'] == store) & (train_df['family'] == fam)]
        sub_test = test_df[(test_df['store_nbr'] == store) & (test_df['family'] == fam)]

        ts_train = sub_train.groupby('date')['sales'].sum().asfreq('D').fillna(0)

        try:
            d = get_diff_order(ts_train)
        except:
            d = 1
        try:
            p, q = get_pq_orders(ts_train)
        except:
            p, q = (1, 1)

        
        try:
            model = ARIMA(ts_train, order=(p,d,q))
            model_fit = model.fit()
        except:
        
            model = ARIMA(ts_train, order=(1,1,1))
            model_fit = model.fit()

        
        steps = len(sub_test['date'].unique())
        forecast = model_fit.forecast(steps=steps)

        
        forecast_df = pd.DataFrame({
            'id': sub_test['id'].values,
            'sales': forecast
        })
        pred_results.append(forecast_df)

# Combine all predictions
final_pred_df = pd.concat(pred_results, ignore_index=True)
final_pred_df = final_pred_df.sort_values('id')


print(final_pred_df.head())

final_pred_df.to_csv("submission.csv", index=False)


#### AIRMA Result : Score: 0.46746